# Module 3: Full Fine-Tuning (SFT) with Vertex AI

In this notebook, we will demonstrate **Full Fine-Tuning (SFT)** on a supported open model (e.g., Llama 3.1 8B or Gemma 3 27B) using Vertex AI.

Unlike LoRA, Full SFT adjusts *all* parameters of the base model. This typically requires more compute and more data, but can yield higher quality results for complex tasks.

We will append this run to our existing **Vertex AI Experiment** to compare its metrics alongside the LoRA job.

In [ ]:
.pip install google-cloud-aiplatform python-dotenv -q

## 1. Setup and Authentication

In [ ]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

PROJECT_ID = os.getenv("PROJECT_ID", "YOUR_PROJECT_ID")
LOCATION = os.getenv("LOCATION", "us-central1") # Tuning is broadly supported in us-central1 and europe-west4
STAGING_BUCKET = os.getenv("STAGING_BUCKET", "YOUR_GCS_BUCKET_NAME")

if not STAGING_BUCKET.startswith("gs://"):
    STAGING_BUCKET = f"gs://{STAGING_BUCKET}"

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Staging Bucket: {STAGING_BUCKET}")

## 2. Re-attach to Vertex Experiment & TensorBoard

In [ ]:
EXPERIMENT_NAME = os.getenv("EXPERIMENT_NAME", "cyber-tuning-demo")
TENSORBOARD_NAME = "cyber-tuning-tb"

# Retrieve existing TensorBoard
tb_instances = aiplatform.Tensorboard.list(filter=f'display_name="{TENSORBOARD_NAME}"')
if tb_instances:
    tb_instance = tb_instances[0]
    print(f"Found TensorBoard: {tb_instance.resource_name}")
    aiplatform.init(experiment=EXPERIMENT_NAME, experiment_tensorboard=tb_instance)
    print(f"Attached to Experiment: {EXPERIMENT_NAME}")
else:
    print("ERROR: TensorBoard instance not found. Run Module 2 first.")

## 3. Configure and Launch Full SFT Job

To perform Full Fine-Tuning in Vertex AI on supported models, we simply omit the `adapter_size` parameter (or use the designated API payload depending on the pipeline template).

In [ ]:
from vertexai.tuning import sft

# ==========================================================================
# PRE-REQUISITE: Paste the SFT_TRAIN_URI from demo_01 output below
SFT_TRAIN_DATA_URI = "gs://YOUR_BUCKET_NAME/tuning_data/your_file.jsonl"
# ==========================================================================

# Must be a supported model, e.g., Llama 3.1 8B
BASE_MODEL = "meta/llama3_1@llama-3.1-8b"

# Define SFT specific hyperparameters
sft_hyperparams = {
    "learning_rate": 2e-5, # Usually lower than LoRA
    "epochs": 3,
}

print("Starting Full Fine-Tuning (SFT) Job..")

# Start a NEW run context in the SAME experiment to compare against LoRA
import datetime
run_name = "run-sft-" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
aiplatform.start_run(run_name)
aiplatform.log_params(sft_hyperparams)
aiplatform.log_params({"tuning_mode": "Full-SFT", "base_model": BASE_MODEL})

try:
    # Note: Omit adapter_size to trigger Full Fine-Tuning.
    # This will spin up A100 or H100 GPU clusters managed by Google Cloud.
    sft_tuning_job = sft.train(
        source_model=BASE_MODEL,
        train_dataset=SFT_TRAIN_DATA_URI,
        
        epochs=sft_hyperparams["epochs"],
        output_uri=f"{STAGING_BUCKET}/tuning_output",
        tuned_model_display_name="cyber-sft-llama",
    )
    
    print("Full SFT job submitted.")
    print(f"Job Name: {sft_tuning_job.name}")
    
except Exception as e:
    print(f"Error submitting job: {e}")

finally:
    aiplatform.end_run()
    print("Experiment run ended.")

## 4. Compare Costs & Metrics

In the Vertex AI console under **Experiments**, you can now select both `run-lora-1` and `run-full-sft-1` and click **Compare**. You will see:
1. Differences in logged Hyperparameters.
2. Separate TensorBoard run trajectories overlaid on each other.